In [1]:
import os
from  dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o", temperature=0)
response = llm.invoke("Hola, soy Ivonne y mi telefono es 1234567890.")
response.text

'Hola Ivonne, es un placer saludarte. Si tienes alguna pregunta o necesitas ayuda con algo, no dudes en decírmelo. Estoy aquí para ayudarte.'

In [3]:
system_prompt = """
Eres un asistente de ventas que ayuda a los clientes a encontrar productos adecuados según sus necesidades.

Tus productos son:
- Computadoras portátiles
- Teléfonos inteligentes
- Auriculares
- Relojes inteligentes
- Tablets
Cuando un cliente te haga una pregunta, responde de manera concisa y amigable, destacando las características clave de los productos que ofreces.
"""
messages = [
    ("system", system_prompt),
    ("user", "Dime los productos que tienes disponibles.")
]

response = llm.invoke(messages)
response.text

'Claro, aquí tienes los productos que ofrecemos:\n\n- **Computadoras portátiles**: Ideales para trabajar y estudiar, con opciones de alto rendimiento y portabilidad.\n- **Teléfonos inteligentes**: Modelos con cámaras de alta calidad, gran capacidad de almacenamiento y baterías duraderas.\n- **Auriculares**: Disponibles en versiones inalámbricas y con cancelación de ruido para una experiencia de sonido envolvente.\n- **Relojes inteligentes**: Perfectos para monitorear tu salud y mantenerte conectado con notificaciones en tiempo real.\n- **Tablets**: Versátiles y ligeras, perfectas para entretenimiento y productividad en movimiento.\n\nSi necesitas más información sobre alguno de estos productos, ¡estaré encantado de ayudarte!'

In [8]:
from langchain_core.tools import tool
import requests

@tool("get_products", description="Obtiene la lista de productos disponibles desde la API de la tienda.")
def get_products():

    # Conectar con API externa (simulada aquí con datos estáticos)
    response = requests.get("https://api.escuelajs.co/api/v1/products")
    products = response.json()
    # return [product for product in products if product["price"] < price]
    return "".join([f"{product['title']} - ${product['price']}\n" for product in products])

In [9]:
get_products.invoke({})

'Classic Red Pullover Hoodie - $10\nClassic Heather Gray Hoodie - $69\nClassic Grey Hooded Sweatshirt - $90\nClassic Black Hooded Sweatshirt - $79\nClassic Comfort Fit Joggers - $25\nClassic Comfort Drawstring Joggers - $79\nClassic Red Jogger Sweatpants - $98\nClassic Navy Blue Baseball Cap - $61\nClassic Blue Baseball Cap - $86\nClassic Red Baseball Cap - $35\nClassic Black Baseball Cap - $58\nClassic Olive Chino Shorts - $84\nClassic High-Waisted Athletic Shorts - $43\nClassic White Crew Neck T-Shirt - $39\nClassic White Tee - Timeless Style and Comfort - $73\nClassic Black T-Shirt - $35\nSleek White & Orange Wireless Gaming Controller - $69\nSleek Wireless Headphone & Inked Earbud Set - $44\nSleek Comfort-Fit Over-Ear Headphones - $28\nEfficient 2-Slice Toaster - $48\nSleek Wireless Computer Mouse - $10\nSleek Modern Laptop with Ambient Lighting - $43\nSleek Modern Laptop for Professionals - $97\nStylish Red & Silver Over-Ear Headphones - $39\nSleek Mirror Finish Phone Case - $27\n

In [10]:
@tool("get_weather", description="Obtiene el clima actual de una ciudad dada.")
def get_weather(city: str) -> str:
    # Llamada a una API de clima
    response = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name=bogota&count=1")
    data = response.json()
    latitude = data['results'][0]['latitude']
    longitude = data['results'][0]['longitude']
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true")
    data = response.json()
    response = f"El clima en {city} es {data['current_weather']['temperature']}°C con vientos de {data['current_weather']['windspeed']} km/h."
    return response

get_weather.invoke({"city": "Bogotá"})

'El clima en Bogotá es 13.8°C con vientos de 1.5 km/h.'

In [13]:
system_prompt = """
Eres un asistente de ventas que ayuda a los clientes a encontrar productos adecuados según sus necesidades y dar el clima de la ciudad

Tus tools son:
- get_products: para obtener los productos que ofreces en la tienda.
- get_weather: para obtener el clima actual de una ciudad dada.
"""
messages = [
    ("system", system_prompt),
    ("user", "Dime los productos que tienes disponibles.")
]
llm_with_tools = llm.bind_tools([get_products, get_weather])
response = llm_with_tools.invoke(messages)
response.tool_calls

[{'name': 'get_products',
  'args': {},
  'id': 'call_Xun0izkxcwDsMroo5HfhK2tQ',
  'type': 'tool_call'}]

In [14]:
messages = [
    ("system", system_prompt),
    ("user", "Hola que tal?")
]
response = llm_with_tools.invoke(messages)
response.text


'¡Hola! ¿En qué puedo ayudarte hoy?'

In [ ]:
system_prompt = """
Eres un asistente de ventas que ayuda a los clientes a encontrar productos adecuados según sus necesidades y dar el clima de la ciudad

Tus tools son:
- get_products: para obtener los productos que ofreces en la tienda.
- get_weather: para obtener el clima actual de una ciudad dada.
"""
messages = [
    ("system", system_prompt),
    ("user", "Cual es el clima en la capital de Nariño?")
]
llm_with_tools = llm.bind_tools([get_products, get_weather])
response = llm_with_tools.invoke(messages)
response.tool_calls

''